# Isolation with Migration (IM) model

Formålet med denne notebook er at udvide coalescent modellen til flere populationer med migration.

IM-modellen beskriver:

1. en ancestral population
2. en split til to populationer
3. mulig migration mellem populationer efter split

Dette er centralt for:

* populationshistorie
* admixture
* fylogenetiske analyser

In [ ]:
# ALWAYS import phasic first
from phasic import Graph, with_ipv, StateIndexer, Property

import numpy as np
import matplotlib.pyplot as plt

from vscodenb import set_vscode_theme
set_vscode_theme()

np.random.seed(42)

## IM-model intuition

Jeg har:

1. to populationer: $pop_1$ og $pop_2$
2. migration mellem dem
3. coalescence inden for hver population

Transitions:

1. Coalescence (samme population)
2. Migration (mellem populationer)

Parametre:

* $\theta_1$,$\theta_2$: coalescent rates
- $m_{12}, m_{21}$: migrationsrater

Dette giver en Markov proces i et udvidet state space.

## State space med populationer 

In [ ]:
nr_samples = 2

indexer = StateIndexer(
    descendants=[
        Property('pop1', min_value=0, max_value=nr_samples),
        Property('pop2', min_value=0, max_value=nr_samples),
        Property('in_pop', min_value=1, max_value=2),
    ]
)

State består af:

* antal lineager i $pop_1$
* antal lineager i $pop_2$
* hvilken population linjen tilhører

## Initial state 

In [ ]:
initial = [0] * indexer.state_length

initial[indexer.descendants.props_to_index(
    pop1=1, pop2=0, in_pop=1
)] = nr_samples

## IM model callback

In [ ]:
@with_ipv(initial)
def IM_model(state, indexer=None):
    transitions = []
    
    if state.sum() <= 1:
        return transitions
    
    for i in range(indexer.descendants.state_length):
        if state[i] == 0:
            continue
        
        props_i = indexer.descendants.index_to_props(i)
        
        for j in range(i, indexer.descendants.state_length):
            if state[j] == 0:
                continue
            
            props_j = indexer.descendants.index_to_props(j)
            
            same = int(i == j)
            
            # Coalescene
            if props_i.in_pop == props_j.in_pop:
                
                if same and state[i] < 2:
                    continue
                if not same and (state[i] < 1 or state[j] < 1):
                    continue
                
                new = state.copy()
                new[i] -= 1
                new[j] -= 1
                
                new_pop1 = props_i.pop1 + props_j.pop1
                new_pop2 = props_i.pop2 + props_j.pop2
                
                new[new_index := indexer.descendants.props_to_index(
                    pop1=new_pop1,
                    pop2=new_pop2,
                    in_pop=props_i.in_pop
                )] += 1
                
                rate = state[i]*(state[j]-same)/(1+same)
                
                transitions.append([new, [rate, 0, 0, 0]])
            
            # Migration
            # pop1 -> pop2
            if props_i.in_pop == 1:
                new = state.copy()
                new[i] -= 1
                
                new[new_index := indexer.descendants.props_to_index(
                    pop1=props_i.pop1,
                    pop2=props_i.pop2,
                    in_pop=2
                )] += 1
                
                transitions.append([new, [0, 1, 0, 0]])
            
            # pop2 -> pop1
            if props_i.in_pop == 2:
                new = state.copy()
                new[i] -= 1
                
                new[new_index := indexer.descendants.props_to_index(
                    pop1=props_i.pop1,
                    pop2=props_i.pop2,
                    in_pop=1
                )] += 1
                
                transitions.append([new, [0, 0, 1, 0]])
    
    return transitions

In [ ]:
graph = Graph(IM_model, indexer=indexer)
graph.plot()

In [ ]:
theta = 1.0
m12 = 0.5
m21 = 0.5

graph.update_weights([theta, m12, m21, 0])

In [ ]:
print("Expectation:", graph.expectation())
print("Variance:", graph.variance())

## Simulation

In [ ]:
samples = graph.sample(1000)

plt.hist(samples, bins=40, density=True)
plt.title("IM model coalescent times")
plt.show()

## Migration effekt

In [ ]:
migration_rates = [0.1, 0.5, 1.0]

for m in migration_rates:
    graph.update_weights([theta, m, m, 0])
    samples = graph.sample(1000)
    
    plt.hist(samples, bins=40, density=True, alpha=0.4, label=f"m={m}")

plt.legend()
plt.title("Effect of migration")
plt.show()

Høj migration $\rightarrow$ populationer opfører sig som en population
Lav migration $\rightarrow$ mere struktur